# Aircraft/Radar Knowledge Graph in Neo4j

This notebook demonstrates how to create the combat-aircraft/radar knowledge graph and load it into Neo4j for graph analytics or downstream r-GCN/Dempster-Shafer experiments.

It uses the repository's `kg_generator.py` as the single source of truth for aircraft variants, radar variants, radar modes, kinematic properties, and operator relationships.


## Prerequisites

1. Start a local Neo4j instance, for example with Docker:

```bash
docker run --rm --name aircraft-kg-neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password \
  neo4j:5
```

2. Install the Neo4j Python driver in the notebook environment if it is not already available:

```bash
pip install neo4j
```

The notebook keeps credentials in variables so they can be changed without editing Cypher.


In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "kg_generator.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from kg_generator import generate_graph, write_json, write_triples

graph = generate_graph()
graph["metadata"]


In [ ]:
output_dir = REPO_ROOT / "generated"
output_dir.mkdir(exist_ok=True)
json_path = output_dir / "aircraft_radar_kg.json"
triples_path = output_dir / "aircraft_radar_triples.csv"

write_json(graph, json_path)
write_triples(graph, triples_path)

print(f"Wrote {json_path}")
print(f"Wrote {triples_path}")


In [ ]:
from collections import Counter

node_labels = Counter(node["label"] for node in graph["nodes"])
edge_types = Counter(edge["relation"] for edge in graph["edges"])

print("Node labels:", dict(node_labels))
print("Relations:", dict(edge_types))


## Connect to Neo4j

The next cell creates a Neo4j driver. Update `NEO4J_URI`, `NEO4J_USER`, and `NEO4J_PASSWORD` if your database uses different settings.


In [ ]:
from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "neo4j"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")


## Create constraints and indexes

Each generated node has a globally unique `id`. The labels are known and finite, so the notebook creates one uniqueness constraint per label.


In [ ]:
CONSTRAINTS = [
    "CREATE CONSTRAINT aircraft_family_id IF NOT EXISTS FOR (n:AircraftFamily) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT aircraft_variant_id IF NOT EXISTS FOR (n:AircraftVariant) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT radar_id IF NOT EXISTS FOR (n:Radar) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT radar_mode_id IF NOT EXISTS FOR (n:RadarMode) REQUIRE n.id IS UNIQUE",
    "CREATE CONSTRAINT operator_id IF NOT EXISTS FOR (n:Operator) REQUIRE n.id IS UNIQUE",
]

with driver.session(database=NEO4J_DATABASE) as session:
    for statement in CONSTRAINTS:
        session.run(statement)

print(f"Created or verified {len(CONSTRAINTS)} constraints")


## Load nodes

Cypher labels cannot be parameterized, so nodes are grouped by known label and loaded with label-specific `MERGE` statements. The generated properties are copied onto the Neo4j nodes with `SET n += row.properties`.


In [ ]:
def load_nodes(tx, label, rows):
    query = f"""
    UNWIND $rows AS row
    MERGE (n:{label} {{id: row.id}})
    SET n += row.properties
    SET n.label = row.label
    RETURN count(n) AS count
    """
    return tx.run(query, rows=rows).single()["count"]

loaded_nodes = {}
with driver.session(database=NEO4J_DATABASE) as session:
    for label in sorted(node_labels):
        rows = [node for node in graph["nodes"] if node["label"] == label]
        loaded_nodes[label] = session.execute_write(load_nodes, label, rows)

loaded_nodes


## Load relationships

Relationship types also cannot be parameterized in Cypher. The cell below groups edges by relation type and then merges them between nodes by their unique `id`.


In [ ]:
def load_edges(tx, relation, rows):
    query = f"""
    UNWIND $rows AS row
    MATCH (source {{id: row.source}})
    MATCH (target {{id: row.target}})
    MERGE (source)-[r:{relation}]->(target)
    RETURN count(r) AS count
    """
    return tx.run(query, rows=rows).single()["count"]

loaded_edges = {}
with driver.session(database=NEO4J_DATABASE) as session:
    for relation in sorted(edge_types):
        rows = [edge for edge in graph["edges"] if edge["relation"] == relation]
        loaded_edges[relation] = session.execute_write(load_edges, relation, rows)

loaded_edges


## Inspect the Neo4j graph

The following queries show simple checks and example traversals that are useful before using the graph for model-building.


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    summary = session.run(
        """
        MATCH (n)
        WITH labels(n)[0] AS label, count(*) AS count
        RETURN label, count
        ORDER BY label
        """
    ).data()

summary


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    examples = session.run(
        """
        MATCH (operator:Operator)-[:OPERATES]->(aircraft:AircraftVariant)-[:USES_RADAR]->(radar:Radar)-[:HAS_MODE]->(mode:RadarMode)
        RETURN operator.name AS operator, aircraft.variant AS aircraft, radar.name AS radar,
               mode.name AS mode, mode.centre_frequency_ghz AS centre_frequency_ghz,
               mode.pulse_repetition_frequency AS prf
        ORDER BY operator, aircraft, radar, mode
        LIMIT 20
        """
    ).data()

examples[:5]


In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    high_speed_interceptors = session.run(
        """
        MATCH (aircraft:AircraftVariant)-[:USES_RADAR]->(radar:Radar)
        WHERE aircraft.max_speed_mach >= 2.5
        RETURN aircraft.variant AS aircraft, aircraft.max_speed_mach AS mach,
               aircraft.service_ceiling_m AS ceiling_m, radar.name AS radar
        ORDER BY mach DESC, aircraft
        """
    ).data()

high_speed_interceptors


## Optional cleanup

Uncomment and run the following cell if you want to remove the imported graph from the Neo4j database.


In [ ]:
# with driver.session(database=NEO4J_DATABASE) as session:
#     session.run("MATCH (n) DETACH DELETE n")
# print("Deleted all nodes and relationships from the selected database")


In [ ]:
driver.close()
